# Noise and the Langevin equation

*others · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/langevin_noise)

A Brownian particle in a harmonic trap is kicked at random by the molecules around it. The Langevin equation adds this noise term, $\sqrt{2\gamma k_BT}\,dW$, to Newton's law; Euler–Maruyama steps a whole ensemble of particles at once in one batch tensor, and the spread of positions settles to the Boltzmann distribution $N(0, k_BT/k)$ (equipartition).

**Vector form**

$$m\,d\mathbf v = \left(-\gamma\,\mathbf v - k\,\mathbf x\right)dt + \sqrt{2\gamma k_BT}\;d\mathbf W,\qquad d\mathbf x = \mathbf v\,dt,\qquad \langle dW\,dW\rangle = dt$$

**Index (tensor) form**

$$v_{b,n+1} = v_{bn} + \frac{-\gamma\,v_{bn} - k\,x_{bn}}{m}\,\Delta t + \frac{\sqrt{2\gamma k_BT\,\Delta t}}{m}\,\xi_{nb} \qquad x_{b,n+1} = x_{bn} + v_{b,n+1}\Delta t \qquad \langle \xi_{nb}\,\xi_{n'b'}\rangle = \delta_{nn'}\delta_{bb'} \qquad \langle x^2\rangle_n = \tfrac1B\,x_{nb}\,x_{nb}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bn},\ v_{bn}$ | `x, v` | (B,) | position and velocity of particle $b$ at step $n$: the whole ensemble is one batch |
| $\xi_{nb}$ | `torch.randn(x.shape, generator=gen)` | (B,) | the noise tensor, one row per step: independent standard normals (fixed seed) |
| $\frac{\sqrt{2\gamma k_BT\Delta t}}{m}$ | `noise_amplitude(gamma, T, m, dt)` | scalar | size of the random kick per step; it scales as $\sqrt{\Delta t}$, not $\Delta t$ |
| $\frac{-\gamma v_{bn} - k x_{bn}}{m}$ | `drift(x, v, gamma, k, m)` | (B,) | deterministic acceleration: friction plus spring |
| $v_{b,n+1},\ x_{b,n+1}$ | `euler_maruyama_step(x, v, xi, ...)` | (B,), (B,) | Euler–Maruyama step; the position uses the new velocity (symplectic Euler) |
| $\tfrac1B\,x_{nb}x_{nb}$ | `einsum('tb,tb->t', x, x) / B` | (T,) | ensemble average: repeated $b$ ⇒ summed over particles |
| $k_BT/k,\ k_BT/m$ | `equipartition(T, k, m)` | scalars | equilibrium $\langle x^2\rangle$ and $\langle v^2\rangle$: $\tfrac12 k\langle x^2\rangle = \tfrac12 m\langle v^2\rangle = \tfrac12 k_BT$ |

Newton's law with friction alone would let the particle come to rest at the bottom of the trap (the bold red curve). Real particles in a fluid never stop: molecules hit them from all sides, and that random force is the noise or error term of the Langevin equation. Its size is not free. The fluctuation–dissipation theorem fixes it to $\sqrt{2\gamma k_BT}$, so the same collisions that cause the friction $\gamma$ also kick the particle, and the ensemble settles at temperature $T$: $\tfrac12 k\langle x^2\rangle = \tfrac12 m\langle v^2\rangle = \tfrac12 k_BT$ (equipartition), with positions distributed as $N(0, k_BT/k)$. The Wiener increment $dW$ has variance $dt$, so in a finite step it becomes $\sqrt{\Delta t}\,\xi$ with $\xi \sim N(0,1)$; the velocity kick per step has variance $2\gamma k_BT\Delta t/m^2$. If the noise were scaled with $\Delta t$ instead, it would vanish as the step shrinks: random kicks add up like a random walk, so their spread grows like $\sqrt{t}$. Because the equation is linear, the ensemble mean $\langle x\rangle$ follows the noise-free damped oscillator exactly (dashed black on red), while each single trajectory wanders. Try $T = 0$ (pure damped oscillator), a large $\gamma$ with a small $m$ (overdamped Brownian motion: $\langle x^2\rangle$ relaxes at rate $2k/\gamma$ without oscillating), or a small $\gamma$ (slow approach to equilibrium). Simplifications: reduced units with $k_B = 1$, one dimension, white (uncorrelated) noise, and a first-order scheme whose equilibrium variances carry an $O(\Delta t)$ bias.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/langevin_noise/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    T, gamma, k, m = params["temperature"], params["gamma"], params["k"], params["m"]
    x0, v0 = tensors.initial_state(params["n_particles"], dev)  # (B,), (B,)
    dt = tensors.time_step(gamma, k, m)

    times, x, v = solve.solve(x0, v0, gamma, k, m, T, params["t_max"], dt, tensors.generator(dev))  # (S,), (S, B)
    x_det = equation.damped_oscillator(times, tensors.X0, gamma, k, m)  # (S,) noise-free curve

    x2 = equation.ensemble_mean_square(x)  # (S,) ⟨x²⟩(t)
    v2 = equation.ensemble_mean_square(v)  # (S,) ⟨v²⟩(t)
    x2_eq, v2_eq = equation.equipartition(T, k, m)
    late = times > 0.5 * times[-1]  # second half of the run: (close to) equilibrium
    print(
        f"B = {x.shape[1]} particles, Δt = {dt:.4g}, noise kick per step σ_v = "
        f"{equation.noise_amplitude(gamma, T, m, dt):.4g}"
    )
    print(f"<x²> late = {x2[late].mean().item():.4f}   (k_B T / k = {x2_eq:.4f})")
    print(f"<v²> late = {v2[late].mean().item():.4f}   (k_B T / m = {v2_eq:.4f})")
    print(f"<x> at end = {x[-1].mean().item():+.4f}   (noise-free x(t_max) = {x_det[-1].item():+.4f})")
    return plot.animate(times, x, x_det, x2, v2, x2_eq, v2_eq, out_dir / "langevin_noise.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/langevin_noise/equation.py
r"""Physics → tensor translation.

A Brownian particle of mass m in a harmonic trap (spring constant k) feels friction -γv from the
surrounding fluid and random kicks from its molecules. The Langevin equation adds this noise (the
"error term") to Newton's law:

    m dv = (-γ v - k x) dt + √(2 γ k_B T) dW,        dx = v dt

dW is a Wiener increment: a Gaussian with mean 0 and variance dt. With a finite step Δt we write
dW = √Δt ξ with ξ ~ N(0, 1). That √Δt (not Δt!) is the key point of stochastic calculus: random kicks
add up like a random walk, so their spread grows like the square root of time.

Euler–Maruyama for a batch of B particles b, time step n, noise tensor ξ_{nb}:

    v_{b,n+1} = v_{bn} + (-γ v_{bn} - k x_{bn}) Δt / m + (√(2 γ k_B T Δt) / m) ξ_{nb}
    x_{b,n+1} = x_{bn} + v_{b,n+1} Δt                          (new velocity: symplectic Euler)

    ⟨ξ_{nb} ξ_{n'b'}⟩ = δ_{nn'} δ_{bb'}      every step and every particle gets an independent kick

Ensemble averages sum over the batch index b:   ⟨x²⟩_n = (1/B) x_{nb} x_{nb}   (b summed).
"""

import math

import torch

K_B = 1.0  # reduced units: energies are measured in units where Boltzmann's constant is 1


def drift(x: torch.Tensor, v: torch.Tensor, gamma: float, k: float, m: float) -> torch.Tensor:
    """Deterministic acceleration a_b = (-γ v_b - k x_b) / m.   x, v: (B,) → (B,)."""
    return (-gamma * v - k * x) / m


def noise_amplitude(gamma: float, T: float, m: float, dt: float) -> float:
    """Standard deviation of the random velocity kick in one step: √(2 γ k_B T Δt) / m.

    Fluctuation–dissipation: the same γ that damps the motion sets the strength of the kicks,
    which is what makes the particle settle at temperature T.
    """
    return math.sqrt(2.0 * gamma * K_B * T * dt) / m


def euler_maruyama_step(x, v, xi, gamma: float, k: float, m: float, T: float, dt: float):
    """One step for every particle at once.   x, v, xi: (B,) → new (x, v), each (B,)."""
    v_new = v + drift(x, v, gamma, k, m) * dt + noise_amplitude(gamma, T, m, dt) * xi
    x_new = x + v_new * dt
    return x_new, v_new


def ensemble_mean_square(x: torch.Tensor) -> torch.Tensor:
    """⟨x²⟩_t = (1/B) Σ_b x_{tb} x_{tb}.   x: (T, B) → (T,);  the batch index b is summed."""
    return torch.einsum("tb,tb->t", x, x) / x.shape[1]


def equipartition(T: float, k: float, m: float) -> tuple[float, float]:
    """Thermal equilibrium: ½ k ⟨x²⟩ = ½ m ⟨v²⟩ = ½ k_B T   →   (⟨x²⟩, ⟨v²⟩) = (k_B T / k, k_B T / m)."""
    return K_B * T / k, K_B * T / m


def gaussian(x: torch.Tensor, variance: float) -> torch.Tensor:
    """Boltzmann distribution in the trap, p(x) ∝ exp(-k x² / 2 k_B T) = N(0, k_B T / k)."""
    return torch.exp(-(x**2) / (2 * variance)) / math.sqrt(2 * math.pi * variance)


def damped_oscillator(t: torch.Tensor, x0: float, gamma: float, k: float, m: float) -> torch.Tensor:
    """Noise-free solution m ẍ = -γ ẋ - k x released from rest at x0 (also the ensemble mean ⟨x⟩(t)).

    x(t) = x0 e^{-βt} (cos ω_d t + (β/ω_d) sin ω_d t),   β = γ / 2m,   ω_d = √(k/m - β²)
    (written with complex ω_d so the same formula covers the overdamped case).
    """
    beta = gamma / (2 * m)
    cdtype = torch.complex128 if t.dtype == torch.float64 else torch.complex64
    wd = torch.sqrt(torch.tensor(k / m - beta**2, dtype=cdtype, device=t.device))
    if wd.abs() < 1e-6:  # critical damping
        return x0 * torch.exp(-beta * t) * (1 + beta * t)
    tc = t.to(cdtype)
    x = x0 * torch.exp(-beta * tc) * (torch.cos(wd * tc) + beta / wd * torch.sin(wd * tc))
    return x.real


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/langevin_noise/tensors.py
"""Build the concrete tensors: the ensemble's initial state and the random-number generator."""

import torch

X0 = 2.0  # every particle is released from rest at x0 = 2, well outside the thermal width
SEED = 0  # fixed seed: the same noise ξ_{nb}, hence the same run, every time


def initial_state(n_particles: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """x_b(0) = x0 and v_b(0) = 0 for every particle b:  two tensors of shape (B,)."""
    x = torch.full((n_particles,), X0, device=device)
    v = torch.zeros(n_particles, device=device)
    return x, v


def generator(device: torch.device, seed: int = SEED) -> torch.Generator:
    """A seeded random-number generator on the same device as the tensors."""
    return torch.Generator(device=device).manual_seed(seed)


def time_step(gamma: float, k: float, m: float) -> float:
    """Δt well below both the damping time m/γ and the oscillation time √(m/k)."""
    return min(0.01, 0.05 * m / gamma, 0.05 * (m / k) ** 0.5)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/langevin_noise/solve.py
"""Euler–Maruyama time stepping of the whole ensemble: one batched step moves all B particles."""

import torch

from problems._common.progress import steps

from .equation import euler_maruyama_step


def solve(
    x0: torch.Tensor,
    v0: torch.Tensor,
    gamma: float,
    k: float,
    m: float,
    T: float,
    t_max: float,
    dt: float,
    gen: torch.Generator,
    n_saved: int = 300,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return (times (S,), x (S, B), v (S, B)), saved at about n_saved evenly spaced times.

    At step n we draw the row ξ_{n,:} of the noise tensor: B independent standard normals.
    """
    n_steps = int(round(t_max / dt))
    save_every = max(1, n_steps // n_saved)
    x, v = x0, v0
    xs, vs, ts = [x], [v], [0.0]
    for n in steps(n_steps, "Euler–Maruyama steps"):
        xi = torch.randn(x.shape, generator=gen, device=x.device)  # ξ_{nb} ~ N(0, 1), shape (B,)
        x, v = euler_maruyama_step(x, v, xi, gamma, k, m, T, dt)
        if (n + 1) % save_every == 0:
            xs.append(x)
            vs.append(v)
            ts.append((n + 1) * dt)
    return torch.tensor(ts, device=x.device), torch.stack(xs), torch.stack(vs)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/langevin_noise/plot.py
"""Animate the ensemble: faint noisy trajectories around the noise-free curve, the histogram of x, and ⟨x²⟩, ⟨v²⟩."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

N_DRAWN = 24  # trajectories drawn faintly (all B particles enter the statistics)
N_BINS = 31


def animate(
    times: torch.Tensor,
    x: torch.Tensor,
    x_det: torch.Tensor,
    x2: torch.Tensor,
    v2: torch.Tensor,
    x2_eq: float,
    v2_eq: float,
    path: Path,
) -> Path:
    t = times.cpu().numpy()
    xs = x.cpu().numpy()  # (S, B)
    xd, m_x2, m_v2 = x_det.cpu().numpy(), x2.cpu().numpy(), v2.cpu().numpy()
    sigma = np.sqrt(x2_eq)
    lim = max(4 * sigma, 1.1 * np.abs(xd).max(), 1e-3)
    edges = np.linspace(-lim, lim, N_BINS + 1)
    width = edges[1] - edges[0]
    counts = np.stack([np.histogram(row, bins=edges)[0] for row in xs]) / (xs.shape[1] * width)  # (S, bins) density

    fig = plt.figure(figsize=(8.2, 4.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.4, 1])
    ax_t = fig.add_subplot(grid[:, 0])
    ax_h = fig.add_subplot(grid[0, 1])
    ax_m = fig.add_subplot(grid[1, 1])

    # --- trajectories x_b(t) ---
    ax_t.set_xlim(0, t[-1])
    ax_t.set_ylim(-lim, lim)
    ax_t.set_xlabel("time t")
    ax_t.set_ylabel("position x")
    ax_t.set_title("Langevin trajectories (noise) vs noise-free")
    if sigma > 0:
        ax_t.axhspan(-sigma, sigma, color="#93c5fd", alpha=0.25, lw=0)
    faint = [ax_t.plot([], [], lw=0.6, color="#2563eb", alpha=0.25)[0] for _ in range(min(N_DRAWN, xs.shape[1]))]
    (bold,) = ax_t.plot([], [], lw=2.4, color="#dc2626", label="noise-free (= mean)")
    (mean,) = ax_t.plot([], [], lw=1.2, color="#111827", ls="--", label=r"ensemble mean $\langle x\rangle$")
    ax_t.legend(loc="upper right", fontsize=8)
    clock = ax_t.text(0.02, 0.04, "", transform=ax_t.transAxes, fontsize=9)

    # --- histogram of x across the ensemble ---
    centers = 0.5 * (edges[1:] + edges[:-1])
    bars = ax_h.bar(centers, counts[0], width=width * 0.9, color="#93c5fd", edgecolor="#2563eb", lw=0.5)
    if sigma > 0:
        fine = np.linspace(-lim, lim, 200)
        ax_h.plot(fine, np.exp(-(fine**2) / (2 * x2_eq)) / np.sqrt(2 * np.pi * x2_eq), color="#dc2626", lw=1.5)
        ax_h.set_ylim(0, 1.6 / np.sqrt(2 * np.pi * x2_eq))
    else:
        ax_h.set_ylim(0, counts.max() * 1.1)
    ax_h.set_xlim(-lim, lim)
    ax_h.set_title(r"histogram of $x_b$ vs $N(0,\,k_BT/k)$", fontsize=10)

    # --- ⟨x²⟩(t) and ⟨v²⟩(t) approaching equipartition ---
    top = 1.15 * max(m_x2.max(), m_v2.max(), 1e-6)
    ax_m.set_xlim(0, t[-1])
    ax_m.set_ylim(0, top)
    ax_m.set_xlabel("time t", labelpad=1)
    ax_m.axhline(x2_eq, color="#2563eb", ls="--", lw=1)
    ax_m.axhline(v2_eq, color="#ea580c", ls=":", lw=1.2)
    (l_x2,) = ax_m.plot([], [], color="#2563eb", lw=1.6, label=r"$\langle x^2\rangle \to k_BT/k$")
    (l_v2,) = ax_m.plot([], [], color="#ea580c", lw=1.6, label=r"$\langle v^2\rangle \to k_BT/m$")
    ax_m.legend(loc="upper right", fontsize=7.5)
    mean_x = xs.mean(axis=1)

    def update(step: int):
        for b, line in enumerate(faint):
            line.set_data(t[: step + 1], xs[: step + 1, b])
        bold.set_data(t[: step + 1], xd[: step + 1])
        mean.set_data(t[: step + 1], mean_x[: step + 1])
        for bar, h in zip(bars, counts[step]):
            bar.set_height(h)
        l_x2.set_data(t[: step + 1], m_x2[: step + 1])
        l_v2.set_data(t[: step + 1], m_v2[: step + 1])
        clock.set_text(f"t = {t[step]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 72), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'temperature': 1.0,
    'gamma': 0.5,
    'k': 1.0,
    'm': 1.0,
    'n_particles': 2000,
    't_max': 40.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.langevin_noise")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.langevin_noise.main")
problem_dir = Path("problems/others/langevin_noise")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
